In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

# 하이퍼파라미터 설정
# 폴더에 이미지가 100장 있다고 가정하면
# 첫 번째 공부: 80명 중 1~32번까지 32장을 가져가서 공부함.

# 두 번째 공부: 그다음 33~64번까지 32장을 가져가서 공부함.

# 세 번째 공부: 남은 **65~80번(16장)**을 가져가서 공부함.
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
DATA_DIR = '../data/dataset_imagenet'

# 이미지 데이터 생성
# 학습 데이터
train_ds = tf.keras.utils.image_dataset_from_directory(
	DATA_DIR,
	validation_split=0.2,
	subset='training',
	seed=1000,#순서정한다
	image_size=IMG_SIZE,
	batch_size=BATCH_SIZE
)

# 검증 데이터
val_ds = tf.keras.utils.image_dataset_from_directory(
	DATA_DIR,
	validation_split=0.2,
	subset='validation',
	seed=1000,
	image_size=IMG_SIZE,
	batch_size=BATCH_SIZE
)

class_names = train_ds.class_names


Found 20 files belonging to 2 classes.
Using 16 files for training.
Found 20 files belonging to 2 classes.
Using 4 files for validation.


In [2]:
# 성능 최적화
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(20).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [3]:
# 모델 설계
# 전이학습 모델 가져옴
base_model = MobileNetV2(input_shape=(224,224,3),include_top=False, weights='imagenet')

# 데이터 증강=>이미지를돌리고 확대하고 움직여서 여러 이미지를 추가하는 작업 
data_augmentation = tf.keras.Sequential([
	layers.RandomFlip('horizontal_and_vertical'),
	layers.RandomRotation(0.2),
	layers.RandomZoom(0.2)
])

# 새 모델 설계
model = models.Sequential([
	layers.Input((224,224,3)),
	data_augmentation, 
	# 정규화를 0~1이 아닌 -1~1로 해야 함. 왜? imagenet에서 그렇게 했기 때문에 맞춰줘야함
	layers.Rescaling(1./127.5, offset=-1), 
	base_model, 
	layers.GlobalAveragePooling2D(),
	layers.Dense(128, activation='relu'),
	layers.Dropout(0.2),
	layers.Dense(2, activation='softmax')
])

In [4]:
# 모델 설정
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
	metrics=['accuracy'])

In [5]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 34s 34s/step - accuracy: 0.4375 - loss: 0.8527 - val_accuracy: 0.5000 - val_loss: 0.9615
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.7500 - loss: 0.3638 - val_accuracy: 0.5000 - val_loss: 0.7837
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.0375 - val_accuracy: 0.7500 - val_loss: 0.6603
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 0.0149 - val_accuracy: 0.7500 - val_loss: 0.7180
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 0.0071 - val_accuracy: 0.7500 - val_loss: 0.8206
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 0.0079 - val_accuracy: 0.7500 - val_loss: 0.9186
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.0022 - val_accuracy: 0.7500 - val_loss: 1.0449
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 0.0014 - val_accuracy: 0.7500 - val_loss: 1.2047
Epoch 9/10
1/1

In [8]:
import requests
import numpy as np
from io import BytesIO
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import mobilenet_v2

def predicts_url(url):
	# 이미지 정보를 가져옴
	response = requests.get(url)
	# 가져온 이미지를 바이트 형태로 변환
	img = image.load_img(BytesIO(response.content), target_size=(224,224))

	# 이미지 전처리 => 입력 형태 맞추기, 이미지를 배열로
	X = image.img_to_array(img)
	X = np.expand_dims(X, axis=0)

	# MobileNetV2전용 전처리 
	# X = mobilenet_v2.preprocess_input(X)

	# 예측
	predicts = model.predict(X)
	
	print('-'*30)
	for i in range(2):
		predict = predicts[0][i]
		print(f'{class_names[i]} : {predict*100:.2f}%')
	print('-'*30)

In [10]:
# 김밥
predicts_url('https://lh4.googleusercontent.com/proxy/YBicREfizdGpPHIZTWBhzPlJ46-6knDM_k2w7ZeilK4ngmnf1CiG3fc500HGTPIEkw3F9ao4lT-TFheDpiDRwBg')
predicts_url('https://image.greating.co.kr/CO/contents/202505/22/58563F38E9F248DB9AABCC37AB246BBC.jpeg')
# 떡볶이
predicts_url('https://cdn.kihoilbo.co.kr/news/photo/202008/880134_302005_3430.png')
predicts_url('https://search.pstatic.net/common/?src=http%3A%2F%2Fcafefiles.naver.net%2FMjAyMDA4MDFfMjEx%2FMDAxNTk2MjE5MDA0Njcw.34rT5-AZ7IrhESUkfln9NZNOZyxsFX0-EBTrZVQLyygg.az20ytNwVDQNCfeL8LrmEBGV1AJnj-rYJgkzlVA3pgwg.JPEG%2FIMG_0355.JPG&type=a340')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
------------------------------
kimbap : 99.98%
tteokbokki : 0.02%
------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
------------------------------
kimbap : 99.61%
tteokbokki : 0.39%
------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
------------------------------
kimbap : 0.02%
tteokbokki : 99.98%
------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
------------------------------
kimbap : 0.00%
tteokbokki : 100.00%
------------------------------
